# 01 — Merge & Extract

Stage 1 of `RESEARCH_PROTOCOL.md`: build one harmonised, patient-level table per
registry (SATI-Q, ANZPICR). We deliberately do **not** force them into one shared
schema — column *names/meanings* are aligned where they overlap, but each registry
keeps its own source columns alongside the harmonised ones.

Outputs (gitignored, local only):
- `derived/sati_q_cohort.parquet`
- `derived/anzpicr_cohort.parquet`

Env: `data_science` conda env.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 60)

SATI_DIR = "SATI-Q_WFPICCS_DATATHON_DATASET"
ANZ_DIR = "ANZPICR_WFPICCS_DATATHON_DATASET"
OUT_DIR = "derived"

import os
os.makedirs(OUT_DIR, exist_ok=True)

## Part 1 — SATI-Q (Argentina)

### 1.1 Load master table + parse dates

In [2]:
fv = pd.read_csv(f"{SATI_DIR}/FIVARAPA_2015_2025.csv", encoding='utf-8-sig')
print("FIVARAPA shape:", fv.shape)

assert fv.duplicated(subset=['TIPODNI', 'DNI', 'FECING']).sum() == 0, \
    "composite key TIPODNI+DNI+FECING is not unique in FIVARAPA"

date_cols = ['FECHAING', 'FECEGR', 'TRAQFI', 'TRAQFF', 'FECINGH', 'FECEGRH']
for c in date_cols:
    fv[c + '_dt'] = pd.to_datetime(fv[c], errors='coerce', dayfirst=True)

fv.head(3)

FIVARAPA shape: (87770, 30)


,TIPODNI,DNI,FECING,FECHAING,REINGRESO,HC,TIPO,PACIENTE,FECEGR,DIAS,EDAD,SEXO,PROCEDENCIA,TRAQ,TRAQI,TRAQE,TRAQFI,TRAQFF,SNG,SNGI,SNGE,ARM,SF,SFI,SFE,FECINGH,FECEGRH,RESTRAT,RESULTADOEGRESOH,Dependencia,FECHAING_dt,FECEGR_dt,TRAQFI_dt,TRAQFF_dt,FECINGH_dt,FECEGRH_dt
0,D.N.I.,@RlvyOzzaiwZPO000002,22/04/2016,22/04/2016,0,HC_000002,0,Nombre_000002,14/05/2016,22,91,M,6,N,N,N,NaN,NaN,S,N,N,S,S,N,N,22/04/2016,NaN,1.0,NaN,Pública,2016-04-22,2016-05-14,NaT,NaT,2016-04-22,NaT
1,D.N.I.,@RlvyOzzaiwZPO000002,15/03/2019,15/03/2019,0,HC_000002,0,Nombre_000002,17/03/2019,2,44,M,1,N,N,N,NaN,NaN,N,N,N,S,N,N,N,15/03/2019,NaN,1.0,NaN,Pública,2019-03-15,2019-03-17,NaT,NaT,2019-03-15,NaT
2,D.N.I.,@RlvyOzzaiwZPO000002,19/10/2022,19/10/2022,0,HC_000002,0,Nombre_000002,24/10/2022,5,88,M,14,N,N,N,NaN,NaN,N,N,N,S,S,N,N,19/10/2022,NaN,1.0,NaN,Pública,2022-10-19,2022-10-24,NaT,NaT,2022-10-19,NaT


### 1.2 Flags, demographics, LOS

`TRAQ`/`TRAQI`/`TRAQE`/`ARM` are coded `'S'`/`'N'` (Spanish Sí/No), not `1`/`0` as
the task brief assumed — confirmed by inspecting `value_counts()` directly, mapped
here explicitly rather than guessed.

`EDAD` units depend on `TIPO` per the SATI-Q data dictionary (months if `TIPO=0`,
years if `TIPO=1`, days if `TIPO=2`). Every row in this extract has `TIPO=0`, so
`EDAD` is in **months** throughout; `age` below is converted to years.

In [3]:
def to_flag(s):
    return (s.astype(str).str.strip().str.upper()
            .map({'S': 1, 'SI': 1, 'N': 0, 'NO': 0, '1': 1, '0': 0, 'TRUE': 1, 'FALSE': 0}))

print("TIPO value_counts (should be all 0 = pediatric, EDAD in months):")
print(fv['TIPO'].value_counts(dropna=False))

fv['TRAQ_flag'] = to_flag(fv['TRAQ'])
fv['TRAQI_flag'] = to_flag(fv['TRAQI'])
fv['TRAQE_flag'] = to_flag(fv['TRAQE'])
fv['mech_vent'] = to_flag(fv['ARM'])

fv['sex'] = fv['SEXO']
fv['age'] = fv['EDAD'] / 12.0  # months -> years, given TIPO==0 for all rows
fv['los_days'] = fv['DIAS']
fv['hosp_los_days'] = (fv['FECEGRH_dt'] - fv['FECINGH_dt']).dt.days
fv['readmission'] = fv['REINGRESO']

n_age_anomaly = ((fv['TIPO'] == 0) & (fv['EDAD'] > 191)).sum()
print(f"\nData-quality flag: {n_age_anomaly} rows have TIPO=0 (pediatric, dictionary range "
      f"1-191 months) but EDAD > 191 months (i.e. age > ~15.9y) — dictionary's stated TIPO=0 "
      f"age range does not hold for all rows. Kept as-is (age = EDAD/12), flagged for awareness.")

TIPO value_counts (should be all 0 = pediatric, EDAD in months):
TIPO
0    87770
Name: count, dtype: int64

Data-quality flag: 1316 rows have TIPO=0 (pediatric, dictionary range 1-191 months) but EDAD > 191 months (i.e. age > ~15.9y) — dictionary's stated TIPO=0 age range does not hold for all rows. Kept as-is (age = EDAD/12), flagged for awareness.


### 1.3 Mortality mapping

**Documented mapping** (SATI-Q data dictionary, `Restrat` tab):

`RESTRAT` is the *ICU* discharge outcome code. Code `5` = Death. Code `0` is
documented as "missing value"; the dictionary defines codes 0–11, but the data
contains one row with code `12` (undocumented — treated as unknown, not death).

`RESULTADOEGRESOH` is a free-text *hospital*-level outcome, populated for a minority
of rows (~24%); `'Fallece'` = died. It can catch a hospital death that happened
*after* ICU discharge (e.g. ICU outcome = "discharged to ward", but died later in
hospital) — RESTRAT alone would miss that, so it's OR'd in.

Final `mortality` = 1 if `RESTRAT==5` OR `RESULTADOEGRESOH` contains `'Fallece'`;
0 if `RESTRAT` is a known non-death code (1–4, 6–11) and `RESULTADOEGRESOH` doesn't
indicate death; otherwise NaN (unknown — `RESTRAT` missing/0/undocumented-code and no
hospital-outcome text to fall back on).

In [4]:
print("RESTRAT value_counts:")
print(fv['RESTRAT'].value_counts(dropna=False).sort_index())
print("\nRESULTADOEGRESOH value_counts (top 10):")
print(fv['RESULTADOEGRESOH'].value_counts(dropna=False).head(10))

KNOWN_NONDEATH_RESTRAT = {1, 2, 3, 4, 6, 7, 8, 9, 10, 11}
died_icu = fv['RESTRAT'] == 5
died_hosp_text = fv['RESULTADOEGRESOH'].str.contains('Fallece', case=False, na=False)
known_alive = fv['RESTRAT'].isin(KNOWN_NONDEATH_RESTRAT) & ~died_hosp_text

fv['mortality'] = np.where(died_icu | died_hosp_text, 1,
                    np.where(known_alive, 0, np.nan))

n_mortality_unknown = fv['mortality'].isna().sum()
print(f"\nmortality is NaN (unknown) for {n_mortality_unknown} rows: RESTRAT missing/0/"
      f"undocumented-code AND RESULTADOEGRESOH doesn't independently indicate death.")
print("mortality value_counts:")
print(fv['mortality'].value_counts(dropna=False))

RESTRAT value_counts:
RESTRAT
0.0      1523
1.0     47963
2.0     14388
3.0        72
4.0      3531
5.0      4569
6.0     10949
7.0       234
8.0      2757
9.0      1168
10.0      577
11.0       33
12.0        1
NaN         5
Name: count, dtype: int64

RESULTADOEGRESOH value_counts (top 10):
RESULTADOEGRESOH
NaN                               66840
Alta domiciliaria                 11747
Fallece                            4640
Se traslada a otra institución     2089
Alta al piso de internación        1108
Homecare                            893
Terapia intermedia                  248
Se traslada a otra UTI               97
Alta voluntaria                      58
 - Seleccionar -                     18
Name: count, dtype: int64



mortality is NaN (unknown) for 1529 rows: RESTRAT missing/0/undocumented-code AND RESULTADOEGRESOH doesn't independently indicate death.
mortality value_counts:
mortality
0.0    81601
1.0     4640
NaN     1529
Name: count, dtype: int64


### 1.4 Severity score — coalesce FiPIM3 (preferred) with FiPim (fallback)

Both files share the `TIPODNI`+`DNI`+`FECING` key with FIVARAPA and have no
duplicate keys. `SCOREPIM3`/`PROBPIM3` (from `FiPIM3`, admissions 2016–2025) are
preferred; `SCOREPIM`/`PROBPIM` (from `FiPim`, admissions 2015–2019, older PIM2
score) are used only where FiPIM3 is missing.

**Harmonisation choice:** both source `PROB*` fields are on a 0–100 percentage
scale. To harmonise with ANZPICR's `PIM3RoD` (a 0–1 probability), `severity_prob`
here is **divided by 100** into a 0–1 probability. `severity_score` (the linear
predictor / log-odds) has no ANZPICR equivalent (see Part 2) and is kept in its
native SATI-Q units.

In [5]:
pim3 = pd.read_csv(f"{SATI_DIR}/FiPIM3_2016_2025.csv", encoding='utf-8-sig')
pim2 = pd.read_csv(f"{SATI_DIR}/FiPim.csv", encoding='utf-8-sig')

key = ['TIPODNI', 'DNI', 'FECING']
assert pim3.duplicated(subset=key).sum() == 0
assert pim2.duplicated(subset=key).sum() == 0

fv = fv.merge(pim3[key + ['SCOREPIM3', 'PROBPIM3', 'ADMISIONELECTIVA']], on=key, how='left',
              suffixes=('', '_pim3'))
fv = fv.merge(pim2[key + ['SCOREPIM', 'PROBPIM', 'ADMISIONELECTIVA']], on=key, how='left',
              suffixes=('', '_pim2'))

fv['severity_score'] = fv['SCOREPIM3'].where(fv['SCOREPIM3'].notna(), fv['SCOREPIM'])
fv['severity_prob'] = (fv['PROBPIM3'].where(fv['PROBPIM3'].notna(), fv['PROBPIM'])) / 100.0
fv['severity_source'] = np.where(fv['SCOREPIM3'].notna(), 'PIM3',
                            np.where(fv['SCOREPIM'].notna(), 'PIM2', 'none'))

# admission category confounder (RESEARCH_PROTOCOL.md covariate list) -- same PIM3/PIM2
# source files also carry ADMISIONELECTIVA (1=elective, 0=non-elective) on the same key,
# free to coalesce alongside severity since both files are already loaded and merged.
fv['elective'] = fv['ADMISIONELECTIVA'].where(fv['ADMISIONELECTIVA'].notna(), fv['ADMISIONELECTIVA_pim2'])

print("severity_source value_counts:")
print(fv['severity_source'].value_counts())
print("\nseverity_prob (0-1) describe:")
print(fv['severity_prob'].describe())
print("\nelective value_counts:")
print(fv['elective'].value_counts(dropna=False))

severity_source value_counts:
severity_source
PIM3    78710
PIM2     7025
none     2035
Name: count, dtype: int64

severity_prob (0-1) describe:
count    85735.000000
mean         0.042317
std          0.106761
min          0.000000
25%          0.002700
50%          0.011300
75%          0.041100
max          0.999900
Name: severity_prob, dtype: float64

elective value_counts:
elective
0.0    60844
1.0    22544
NaN     4382
Name: count, dtype: int64


### 1.4a "Cost of waiting" complications (`FiCompUti`)

Per `RESEARCH_PROTOCOL.md`'s Framing section: device/immobility-time-dependent harms
that accrue the longer a child stays intubated *without* a trach. Five flags, joined
on the same composite key: `vap`=`NAR`, `accidental_extubation`=
`EXTUBACION_ACCIDENTAL`, `clabsi`=`BACTAC`, `cauti`=`UTI`, `pressure_ulcer`=`ESCARAS`.

**Coding direction, verified empirically:** the SATI-Q data dictionary states these
fields are `0=Yes, 1=No`. That's wrong for this export — `1` occurs in only 1–2.3%
of admissions across all five fields (matching plausible real-world incidence);
treating `0` as "Yes" would imply ~98% of admissions had VAP, which is absurd.
**Standard coding used here: `1`=Yes/event happened, `0`=No — not inverted.**

**Duplicate keys:** `FiCompUti` has 323 admissions with >1 row on the composite
key (9 of those have a flag that actually disagrees between the duplicate rows).
Aggregated with `max()` per flag within each key before merging, so "the event
happened" wins over "it didn't" when duplicate records disagree — the conservative
direction for a complication flag.

In [6]:
comp = pd.read_csv(f"{SATI_DIR}/FiCompUti_2015_2025.csv", encoding='utf-8-sig')
comp_flags = {'NAR': 'vap', 'EXTUBACION_ACCIDENTAL': 'accidental_extubation',
              'BACTAC': 'clabsi', 'UTI': 'cauti', 'ESCARAS': 'pressure_ulcer'}

n_dupe_keys = comp.duplicated(subset=key).sum()
print(f"FiCompUti duplicate composite-key rows: {n_dupe_keys}")
for c in comp_flags:
    print(f"{c} value_counts:")
    print(comp[c].value_counts(dropna=False))

comp_agg = comp.groupby(key, as_index=False)[list(comp_flags)].max()
comp_agg = comp_agg.rename(columns=comp_flags)
fv = fv.merge(comp_agg, on=key, how='left')

print("\nharmonised complication rates (% of all admissions with a FiCompUti record):")
print((fv[list(comp_flags.values())].mean() * 100).round(2))

FiCompUti duplicate composite-key rows: 323
NAR value_counts:
NAR
0    86818
1     2047
Name: count, dtype: int64
EXTUBACION_ACCIDENTAL value_counts:
EXTUBACION_ACCIDENTAL
0    87443
1     1422
Name: count, dtype: int64
BACTAC value_counts:
BACTAC
0    87891
1      974
Name: count, dtype: int64
UTI value_counts:
UTI
0    87372
1     1493
Name: count, dtype: int64
ESCARAS value_counts:
ESCARAS
0    87903
1      962
Name: count, dtype: int64

harmonised complication rates (% of all admissions with a FiCompUti record):
vap                      2.28
accidental_extubation    1.60
clabsi                   1.09
cauti                    1.67
pressure_ulcer           1.08
dtype: float64


### 1.5 Diagnosis group (bonus, supports Task 2 §4)

`FiMotingP` gives one standardised diagnosis category (`PATING`) per admission on
the same composite key, no dupes — a clean confounder for "sicker kids trached
earlier/later" and case-mix comparisons. Not in the original task brief's column
list, but directly requested by `RESEARCH_PROTOCOL.md`'s covariate list and Task 2
§4, and free to add given it joins cleanly.

In [7]:
patp = pd.read_csv(f"{SATI_DIR}/FiMotingP_2015_2025.csv", encoding='utf-8-sig')
assert patp.duplicated(subset=key).sum() == 0
fv = fv.merge(patp[key + ['PATING']].rename(columns={'PATING': 'diagnosis_group'}), on=key, how='left')
print(fv['diagnosis_group'].value_counts(dropna=False))

diagnosis_group
Respiratorio      29628
Postquirúrgico    19594
Otros             13106
NaN                8595
Neurológico        8050
Causa Externa      5552
Cardiológico       3245
Name: count, dtype: int64


### 1.6 Timing cohort

Per `RESEARCH_PROTOCOL.md` / task brief: new tracheostomy = `TRAQ='S'` AND
`TRAQI='N'` (used during stay, not present on admission). Eligibility for the
*timing* analysis additionally requires both `TRAQFI` and `TRAQFF` non-null and
`days_to_trach >= 0` (drops the known negative-duration data-entry errors).

Rows failing eligibility are **kept** in the output (not silently dropped) with
`eligible_for_timing_analysis=False`, so later stages can decide whether/how to use
them (e.g. `days_to_trach` alone, without needing `TRAQFF`).

In [8]:
fv['days_to_trach'] = (fv['TRAQFI_dt'] - fv['FECHAING_dt']).dt.days

new_trach_mask = (fv['TRAQ_flag'] == 1) & (fv['TRAQI_flag'] == 0)
both_dates = fv['TRAQFI_dt'].notna() & fv['TRAQFF_dt'].notna()
nonneg = fv['days_to_trach'] >= 0

fv['eligible_for_timing_analysis'] = new_trach_mask & both_dates & nonneg

n_total = len(fv)
n_trach = (fv['TRAQ_flag'] == 1).sum()
n_new = new_trach_mask.sum()
n_missing_dates = (new_trach_mask & ~both_dates).sum()
n_negative = (new_trach_mask & both_dates & ~nonneg).sum()
n_eligible = fv['eligible_for_timing_analysis'].sum()

print(f"total admissions:                         {n_total:,}")
print(f"trach admissions (TRAQ=S):                 {n_trach:,}")
print(f"new trach (TRAQ=S & TRAQI=N):               {n_new:,}")
print(f"  -> missing TRAQFI and/or TRAQFF:          {n_missing_dates:,}")
print(f"  -> negative duration (TRAQFI<FECHAING):   {n_negative:,}  <- dropped from eligible")
print(f"eligible_for_timing_analysis:                {n_eligible:,}")

extreme = fv.loc[new_trach_mask & both_dates & nonneg & (fv['days_to_trach'] > 365),
                  ['DNI', 'FECING', 'FECHAING', 'TRAQFI', 'days_to_trach', 'los_days']]
if len(extreme):
    print(f"\nData-quality flag: {len(extreme)} 'eligible' rows have days_to_trach > 365 "
          f"days (plausible for very long stays, but worth a second look in Task 2's "
          f"days_to_trach<=los_days cross-check):")
    print(extreme.to_string(index=False))

impossible = fv.loc[fv['eligible_for_timing_analysis'] & (fv['days_to_trach'] > fv['los_days']),
                     ['DNI', 'FECHAING', 'TRAQFI', 'days_to_trach', 'los_days']]
print(f"\nData-quality flag: {len(impossible)} eligible rows have days_to_trach > los_days "
      f"(placement recorded after the whole PICU stay ended -- impossible; one is the "
      f"same TRAQFI-year-2109 typo above). Not excluded here (Task 1 keeps the row + flag "
      f"per the brief), but Task 2's own days_to_trach<=los_days cross-check should catch "
      f"these too:")
print(impossible.to_string(index=False))

total admissions:                         87,770
trach admissions (TRAQ=S):                 4,413
new trach (TRAQ=S & TRAQI=N):               2,021
  -> missing TRAQFI and/or TRAQFF:          881
  -> negative duration (TRAQFI<FECHAING):   4  <- dropped from eligible
eligible_for_timing_analysis:                1,136

Data-quality flag: 1 'eligible' rows have days_to_trach > 365 days (plausible for very long stays, but worth a second look in Task 2's days_to_trach<=los_days cross-check):
                 DNI     FECING   FECHAING     TRAQFI  days_to_trach  los_days
VrB5TPMk2BV@cI000633 31/01/2019 31/01/2019 26/02/2109        32898.0        50

Data-quality flag: 3 eligible rows have days_to_trach > los_days (placement recorded after the whole PICU stay ended -- impossible; one is the same TRAQFI-year-2109 typo above). Not excluded here (Task 1 keeps the row + flag per the brief), but Task 2's own days_to_trach<=los_days cross-check should catch these too:
                 DNI   FECHAIN

### 1.7 Assemble SATI-Q output table

Output = one row per **new-trach admission** (`TRAQ=S & TRAQI=N`, n as above) — not
just the timing-eligible subset. Pre-existing-trach admissions (`TRAQI=S`) are out
of scope for this cohort entirely (per protocol's cohort definition) and are not
included.

In [9]:
id_cols = ['TIPODNI', 'DNI', 'FECING']
keep_cols = id_cols + [
    'registry', 'age', 'sex', 'los_days', 'hosp_los_days', 'mech_vent', 'mortality',
    'readmission', 'severity_score', 'severity_prob', 'severity_source', 'elective',
    'vap', 'accidental_extubation', 'clabsi', 'cauti', 'pressure_ulcer',
    'SCOREPIM3', 'PROBPIM3', 'SCOREPIM', 'PROBPIM',
    'diagnosis_group', 'days_to_trach', 'eligible_for_timing_analysis',
    'RESTRAT', 'RESULTADOEGRESOH', 'TRAQFI', 'TRAQFF', 'FECHAING', 'DIAS',
]

fv['registry'] = 'SATI-Q'
sati_q_cohort = fv.loc[new_trach_mask, keep_cols].reset_index(drop=True)
print("sati_q_cohort shape:", sati_q_cohort.shape)
sati_q_cohort.head()

sati_q_cohort shape: (2021, 33)


,TIPODNI,DNI,FECING,registry,age,sex,los_days,hosp_los_days,mech_vent,mortality,readmission,severity_score,severity_prob,severity_source,elective,vap,accidental_extubation,clabsi,cauti,pressure_ulcer,SCOREPIM3,PROBPIM3,SCOREPIM,PROBPIM,diagnosis_group,days_to_trach,eligible_for_timing_analysis,RESTRAT,RESULTADOEGRESOH,TRAQFI,TRAQFF,FECHAING,DIAS
0,D.N.I.,@RlvyOzzaiwZPO000011,21/05/2016,SATI-Q,0.833333,F,4,NaN,0.0,1.0,1,1.08,0.7460,PIM3,0.0,1.0,0.0,0.0,0.0,0.0,1.08,74.60,0.06,51.50,Respiratorio,NaN,False,5.0,Fallece,NaN,NaN,21/05/2016,4
1,D.N.I.,@RlvyOzzaiwZPO000019,02/04/2019,SATI-Q,0.333333,F,37,NaN,1.0,0.0,0,0.00,0.0000,PIM3,0.0,0.0,0.0,0.0,1.0,0.0,0.00,0.00,NaN,NaN,Respiratorio,27.0,True,6.0,NaN,29/04/2019,09/05/2019,02/04/2019,37
2,D.N.I.,@RlvyOzzaiwZPO000058,13/05/2019,SATI-Q,0.333333,F,42,NaN,1.0,0.0,0,-5.26,0.0052,PIM3,0.0,0.0,1.0,0.0,1.0,0.0,-5.26,0.52,NaN,NaN,Respiratorio,32.0,True,6.0,NaN,14/06/2019,24/06/2019,13/05/2019,42
3,D.N.I.,@RlvyOzzaiwZPO000068,24/05/2019,SATI-Q,3.333333,M,30,NaN,1.0,0.0,0,-2.94,0.0502,PIM3,0.0,1.0,0.0,0.0,0.0,0.0,-2.94,5.02,NaN,NaN,Respiratorio,19.0,True,6.0,NaN,12/06/2019,24/06/2019,24/05/2019,30
4,D.N.I.,@RlvyOzzaiwZPO000075,03/08/2017,SATI-Q,2.500000,M,26,NaN,1.0,0.0,0,-5.59,0.0037,PIM3,0.0,0.0,0.0,0.0,0.0,0.0,-5.59,0.37,-5.13,0.59,Respiratorio,13.0,True,1.0,NaN,16/08/2017,29/08/2017,03/08/2017,26


## Part 2 — ANZPICR (Australia / New Zealand)

### 2.1 Load admissions table (selected columns only — 59MB file)

`ANZPICR_ADM.csv` already ships ready-made `ICUAdmDHr`/`ICUDisDHr`/`HospAdmDHr`/
`HospDisDHr` datetime-string columns (`YYYY-MM-DD HH:MM:SS`), confirmed complete
(0 nulls) in this file — used directly instead of manually re-assembling from the
Year/Month/Day/Hour parts, which is simpler and avoids reconstruction bugs; the
task brief didn't know these existed when it was written.

In [10]:
usecols = [
    'DSiteID', 'ICU_NO', 'DURNo', 'SEX', 'AGE',
    'ICUAdmYear', 'ICUAdmMonth', 'ICUAdmDay', 'ICUAdmHour', 'ICUAdmDHr',
    'ICUDisYear', 'ICUDisMonth', 'ICUDisDay', 'ICUDisHour', 'ICUDisDHr',
    'ICU_HRS', 'OUTCOME', 'DIED_IN_ICU', 'DIED_IN_HOSP', 'READMITTED', 'ELECTIVE', 'PDX',
    'PIM3_LR', 'PIM3_HR', 'PIM3_VHR', 'PIM3RoD', 'MECHVENT_HRS', 'INV_HRS', 'RS_HR124',
    'HospAdmYear', 'HospAdmMonth', 'HospAdmDay', 'HospAdmHour', 'HospAdmDHr',
    'HospDisYear', 'HospDisMonth', 'HospDisDay', 'HospDisHour', 'HospDisDHr',
]
adm = pd.read_csv(f"{ANZ_DIR}/ANZPICR_ADM.csv", usecols=usecols, encoding='utf-8-sig')
print("ANZPICR_ADM shape:", adm.shape)

key = ['DSiteID', 'ICU_NO', 'DURNo']
assert adm.duplicated(subset=key).sum() == 0, "composite key not unique in ANZPICR_ADM"

for c in ['ICUAdmDHr', 'ICUDisDHr', 'HospAdmDHr', 'HospDisDHr']:
    adm[c + '_dt'] = pd.to_datetime(adm[c], errors='coerce')
    print(f"{c}: {adm[c].isna().sum()} null of {len(adm)}")

ANZPICR_ADM shape: (122622, 39)
ICUAdmDHr: 0 null of 122622
ICUDisDHr: 0 null of 122622
HospAdmDHr: 0 null of 122622
HospDisDHr: 275 null of 122622


### 2.2 Flags, demographics, LOS, severity

**Mortality:** `DIED_IN_ICU` (`OUTCOME` in {2, 5}) and `DIED_IN_HOSP`
(`HOSP_OUT`==2) are pre-derived, complete (no nulls) — `mortality` = either flag.

**mech_vent:** `INV_HRS` (invasive ventilation hours; dictionary states
`0.00 or NULL` both mean *no* invasive ventilation) chosen over `RS_HR124`, because
`RS_HR124` only captures the first hour of admission and includes non-invasive
support — `ARM` on the SATI-Q side means invasive ventilation *at any point during
the stay*, which `INV_HRS>0` matches much more closely.

**severity_prob:** `PIM3RoD` is already a 0–1 probability — used as-is (see §1.4
for why SATI-Q's percentage-scale fields are divided by 100 to match).
**severity_score:** ANZPICR's admissions file exposes only the resulting
probability, not the underlying linear predictor — no equivalent to SATI-Q's
`SCOREPIM3` is available, so `severity_score` is left `NaN` for this registry.

**readmission:** `READMITTED` is coded `0`/`1` per the data dictionary, but the
actual data uses blank/NaN for the "no" case (never a literal `0`) plus 5 rows with
an undocumented code `2`. Mapped as: `1` if `READMITTED==1`; `0` if NaN (treated as
the dictionary's default "not a readmission"); `NaN` (flagged) for the 5 rows coded
`2`.

In [11]:
adm['sex'] = adm['SEX']
adm['age'] = adm['AGE']
adm['los_days'] = adm['ICU_HRS'] / 24.0
adm['hosp_los_days'] = (adm['HospDisDHr_dt'] - adm['HospAdmDHr_dt']).dt.total_seconds() / 86400.0
adm['mech_vent'] = (adm['INV_HRS'].fillna(0) > 0).astype(int)
adm['mortality'] = ((adm['DIED_IN_ICU'] == 1) | (adm['DIED_IN_HOSP'] == 1)).astype(int)
adm['severity_prob'] = adm['PIM3RoD']
adm['severity_score'] = np.nan
adm['severity_source'] = 'PIM3RoD'

print('READMITTED value_counts:')
print(adm['READMITTED'].value_counts(dropna=False))
n_unmapped_readmit = (adm['READMITTED'] == 2).sum()
adm['readmission'] = np.where(adm['READMITTED'] == 1, 1,
                        np.where(adm['READMITTED'] == 2, np.nan, 0))
print(f"\nData-quality flag: {n_unmapped_readmit} rows have READMITTED==2 "
      f"(undocumented code, not in the ANZPICR data dictionary's 0/1 scheme) "
      f"-> mapped to NaN, not silently coerced to 0 or 1.")
print('\nharmonised readmission value_counts:')
print(adm['readmission'].value_counts(dropna=False))

n_age_anomaly = (adm['AGE'] > 18).sum()
print(f"\nData-quality flag: {n_age_anomaly} rows have AGE > 18 years, despite the "
      f"registry documentation describing this as a PICU (<=18y) / general-ICU (<16y) "
      f"cohort. Kept as-is, flagged for awareness.")
print(f"SEX value_counts (I/U are non-M/F codes introduced 2017 per dictionary):")
print(adm['SEX'].value_counts(dropna=False))

READMITTED value_counts:
READMITTED
NaN    113633
1.0      8984
2.0         5
Name: count, dtype: int64

Data-quality flag: 5 rows have READMITTED==2 (undocumented code, not in the ANZPICR data dictionary's 0/1 scheme) -> mapped to NaN, not silently coerced to 0 or 1.

harmonised readmission value_counts:
readmission
0.0    113633
1.0      8984
NaN         5
Name: count, dtype: int64

Data-quality flag: 98 rows have AGE > 18 years, despite the registry documentation describing this as a PICU (<=18y) / general-ICU (<16y) cohort. Kept as-is, flagged for awareness.
SEX value_counts (I/U are non-M/F codes introduced 2017 per dictionary):
SEX
M    69236
F    53341
I       30
U       15
Name: count, dtype: int64


### 2.3 New-tracheostomy timing cohort

`ANZPICR_DIAG` diagnosis code `1506` = tracheostomy. `ADX_CAT==3` ("ICU occurrence")
is the only category with a mandatory timestamp (`ADX_DHr`) and the only one
analogous to SATI-Q's "new, not present on admission" (`TRAQI=0`); `ADX_CAT` 1/2
(pre-existing/acute-on-admission) and 9 (not coded) are excluded from the timing
cohort, matching the exclusion of SATI-Q's `TRAQI=1` patients.

In [12]:
diag = pd.read_csv(f"{ANZ_DIR}/ANZPICR_DIAG.csv", encoding='utf-8-sig')
print("ANZPICR_DIAG shape:", diag.shape)

trach = diag[diag['ADX'] == 1506].copy()
print("\nADX_CAT value_counts among ADX=1506 rows:")
print(trach['ADX_CAT'].value_counts(dropna=False))

new_trach = trach[trach['ADX_CAT'] == 3].copy()
n_dupe_keys = new_trach.duplicated(subset=key).sum()
if n_dupe_keys:
    print(f"\n{n_dupe_keys} admission(s) have >1 new-trach diagnosis row — "
          f"deduping by earliest ADX_DHr:")
    print(new_trach[new_trach.duplicated(subset=key, keep=False)].sort_values(key))
new_trach = new_trach.sort_values('ADX_DHr').drop_duplicates(subset=key, keep='first')

new_trach['ADX_DHr_dt'] = pd.to_datetime(new_trach['ADX_DHr'], errors='coerce')
print(f"\nADX_DHr null among ADX_CAT==3 rows: {new_trach['ADX_DHr_dt'].isna().sum()}")

ANZPICR_DIAG shape: (263698, 6)

ADX_CAT value_counts among ADX=1506 rows:
ADX_CAT
9    261
3    176
1     33
2     20
Name: count, dtype: int64

1 admission(s) have >1 new-trach diagnosis row — deduping by earliest ADX_DHr:
        DSiteID    ICU_NO    DURNo   ADX  ADX_CAT                  ADX_DHr
104039      858  20220638  HB@AEGB  1506        3  2022-09-24 13:00:00.000
104040      858  20220638  HB@AEGB  1506        3  2022-07-15 09:00:00.000

ADX_DHr null among ADX_CAT==3 rows: 0


In [13]:
anz_trach = new_trach.merge(adm, on=key, how='left', indicator=True)
n_unmatched = (anz_trach['_merge'] == 'left_only').sum()
print(f"new-trach diagnosis rows with no matching ADM admission: {n_unmatched}")
anz_trach = anz_trach.drop(columns=['_merge'])

anz_trach['days_to_trach'] = (
    (anz_trach['ADX_DHr_dt'] - anz_trach['ICUAdmDHr_dt']).dt.total_seconds() / 86400.0
)

nonneg = anz_trach['days_to_trach'] >= 0
has_dt = anz_trach['ADX_DHr_dt'].notna() & anz_trach['ICUAdmDHr_dt'].notna()
anz_trach['eligible_for_timing_analysis'] = has_dt & nonneg

n_total = len(adm)
n_trach = trach.drop_duplicates(subset=key).shape[0]
n_new = len(anz_trach)
n_negative = (has_dt & ~nonneg).sum()
n_eligible = anz_trach['eligible_for_timing_analysis'].sum()

print(f"\ntotal ICU admissions:                       {n_total:,}")
print(f"admissions with any trach diagnosis (dedup): {n_trach:,}")
print(f"new trach (ADX_CAT==3, dedup):                {n_new:,}")
print(f"  -> missing ADX_DHr and/or ICUAdmDHr:        {(~has_dt).sum():,}")
print(f"  -> negative duration:                       {n_negative:,}  <- dropped from eligible")
print(f"eligible_for_timing_analysis:                  {n_eligible:,}")

new-trach diagnosis rows with no matching ADM admission: 0

total ICU admissions:                       122,622
admissions with any trach diagnosis (dedup): 488
new trach (ADX_CAT==3, dedup):                175
  -> missing ADX_DHr and/or ICUAdmDHr:        0
  -> negative duration:                       0  <- dropped from eligible
eligible_for_timing_analysis:                  175


### 2.4 Assemble ANZPICR output table

In [14]:
anz_trach['registry'] = 'ANZPICR'
id_cols = ['DSiteID', 'ICU_NO', 'DURNo']
keep_cols = id_cols + [
    'registry', 'age', 'sex', 'los_days', 'hosp_los_days', 'mech_vent', 'mortality',
    'readmission', 'severity_score', 'severity_prob', 'severity_source',
    'PIM3RoD', 'PIM3_LR', 'PIM3_HR', 'PIM3_VHR', 'PDX', 'ELECTIVE',
    'MECHVENT_HRS', 'INV_HRS', 'days_to_trach', 'eligible_for_timing_analysis',
    'DIED_IN_ICU', 'DIED_IN_HOSP', 'OUTCOME', 'ADX_DHr', 'ICUAdmDHr', 'ICU_HRS',
]
anzpicr_cohort = anz_trach[keep_cols].reset_index(drop=True)
print("anzpicr_cohort shape:", anzpicr_cohort.shape)
anzpicr_cohort.head()

anzpicr_cohort shape: (175, 30)


,DSiteID,ICU_NO,DURNo,registry,age,sex,los_days,hosp_los_days,mech_vent,mortality,readmission,severity_score,severity_prob,severity_source,PIM3RoD,PIM3_LR,PIM3_HR,PIM3_VHR,PDX,ELECTIVE,MECHVENT_HRS,INV_HRS,days_to_trach,eligible_for_timing_analysis,DIED_IN_ICU,DIED_IN_HOSP,OUTCOME,ADX_DHr,ICUAdmDHr,ICU_HRS
0,858,20191514,F?HB@AB,ANZPICR,0,F,16.295000,249.625000,1,0,1.0,NaN,0.033567,PIM3RoD,0.033567,0,0,0,469,0,223.67,176.83,0.0,True,0,0,1,2019-12-02 07:00:00.000,2019-12-02 07:00:00.000,391.08
1,637,20200289,@AHG?@F,ANZPICR,0,F,489.869583,489.875000,1,0,0.0,NaN,0.031660,PIM3RoD,0.031660,0,0,0,461,0,11754.37,11754.37,64.0,True,0,0,1,2020-05-06 14:00:00.000,2020-03-03 14:00:00.000,11756.87
2,858,20200591,H?AEGBC,ANZPICR,0,M,2.509583,2.708333,1,1,0.0,NaN,0.402802,PIM3RoD,0.402802,0,0,1,802,0,56.50,56.50,0.0,True,1,1,2,2020-06-04 05:00:00.000,2020-06-04 05:00:00.000,60.23
3,858,20200873,H?BFEEE,ANZPICR,0,F,8.736250,23.666667,1,0,1.0,NaN,0.017467,PIM3RoD,0.017467,0,0,0,410,0,109.42,106.42,0.0,True,0,0,1,2020-09-02 17:00:00.000,2020-09-02 17:00:00.000,209.67
4,858,20200885,@AEHHGH,ANZPICR,11,F,85.934583,182.250000,1,0,0.0,NaN,0.056227,PIM3RoD,0.056227,0,0,0,102,0,1244.05,1224.05,0.0,True,0,0,1,2020-09-05 16:00:00.000,2020-09-05 16:00:00.000,2062.43


## Part 3 — Output summary

In [15]:
print("=" * 70)
print("EXCLUSION FUNNEL SUMMARY")
print("=" * 70)
print(f"{'Step':<45}{'SATI-Q':>12}{'ANZPICR':>12}")
print(f"{'Total PICU admissions':<45}{len(fv):>12,}{len(adm):>12,}")
sati_n_trach = (fv['TRAQ_flag'] == 1).sum()
sati_n_new = new_trach_mask.sum()
sati_n_eligible = int(fv['eligible_for_timing_analysis'].sum())
anz_n_trach = n_trach
anz_n_new = n_new
anz_n_eligible = int(n_eligible)
print(f"{'Any tracheostomy':<45}{sati_n_trach:>12,}{anz_n_trach:>12,}")
print(f"{'New tracheostomy (not pre-existing)':<45}{sati_n_new:>12,}{anz_n_new:>12,}")
print(f"{'Eligible for timing analysis':<45}{sati_n_eligible:>12,}{anz_n_eligible:>12,}")

print()
print("=" * 70)
print("DATA-QUALITY / PROTOCOL-MISMATCH FLAGS")
print("=" * 70)
flags = [
    "SATI-Q TRAQ/TRAQI/TRAQE/ARM are 'S'/'N' strings, not 1/0 as the task brief assumed.",
    "SATI-Q EDAD units depend on TIPO (months/years/days); all rows here are TIPO=0 "
    "(months), but ~1,316 rows have EDAD>191 months, outside the dictionary's stated "
    "TIPO=0 range (1-191 months) -- age derived as EDAD/12 regardless, flagged not fixed.",
    "SATI-Q RESTRAT has one row with undocumented code 12 (dictionary defines 0-11).",
    "SATI-Q: 4 'new trach' rows have days_to_trach > 365d after both-dates filtering; "
    "one of these (TRAQFI year 2109) is almost certainly a data-entry typo -- not "
    "excluded here (Task 1 only drops negatives per the brief), flagged for Task 2's "
    "days_to_trach<=los_days cross-check.",
    "ANZPICR READMITTED uses blank/NaN for the documented '0=No' case (never a literal "
    "0) and has 5 rows with an undocumented code 2 -> mapped to NaN.",
    f"ANZPICR AGE has {int((adm['AGE'] > 18).sum())} rows > 18 years despite registry "
    "documentation describing this as a <=18y (PICU) / <16y (general ICU) cohort.",
    "ANZPICR SEX includes small counts of 'I'/'U' codes alongside M/F.",
    "ANZPICR has no equivalent to SATI-Q's SCOREPIM3 linear predictor -- only the "
    "resulting probability (PIM3RoD) is exposed in ANZPICR_ADM, so severity_score is "
    "NaN for the entire ANZPICR cohort.",
    "SATI-Q FiCompUti (complications) data dictionary claims 0=Yes/1=No; verified "
    "empirically backwards (1 occurs in 1-2.3% of admissions per field, matching real "
    "incidence) -- used 1=Yes as documented in RESEARCH_PROTOCOL.md's corrected Framing "
    "section. FiCompUti also has 323 duplicate composite-key rows (9 with disagreeing "
    "flag values) -- aggregated via max() per flag before merging.",
    "No ANZPICR equivalent to SATI-Q's FiCompUti complications file has been identified "
    "-- vap/accidental_extubation/clabsi/cauti/pressure_ulcer are SATI-Q-only columns "
    "(NaN for the whole ANZPICR cohort). Left as an open item per RESEARCH_PROTOCOL.md.",
]
for i, flag in enumerate(flags, 1):
    print(f"{i}. {flag}\n")

EXCLUSION FUNNEL SUMMARY
Step                                               SATI-Q     ANZPICR
Total PICU admissions                              87,770     122,622
Any tracheostomy                                    4,413         488
New tracheostomy (not pre-existing)                 2,021         175
Eligible for timing analysis                        1,136         175

DATA-QUALITY / PROTOCOL-MISMATCH FLAGS
1. SATI-Q TRAQ/TRAQI/TRAQE/ARM are 'S'/'N' strings, not 1/0 as the task brief assumed.

2. SATI-Q EDAD units depend on TIPO (months/years/days); all rows here are TIPO=0 (months), but ~1,316 rows have EDAD>191 months, outside the dictionary's stated TIPO=0 range (1-191 months) -- age derived as EDAD/12 regardless, flagged not fixed.

3. SATI-Q RESTRAT has one row with undocumented code 12 (dictionary defines 0-11).

4. SATI-Q: 4 'new trach' rows have days_to_trach > 365d after both-dates filtering; one of these (TRAQFI year 2109) is almost certainly a data-entry typo -- not excl

In [16]:
sati_q_cohort.to_parquet(f"{OUT_DIR}/sati_q_cohort.parquet", index=False)
anzpicr_cohort.to_parquet(f"{OUT_DIR}/anzpicr_cohort.parquet", index=False)
print(f"Wrote {OUT_DIR}/sati_q_cohort.parquet  ({sati_q_cohort.shape[0]} rows, {sati_q_cohort.shape[1]} cols)")
print(f"Wrote {OUT_DIR}/anzpicr_cohort.parquet ({anzpicr_cohort.shape[0]} rows, {anzpicr_cohort.shape[1]} cols)")

Wrote derived/sati_q_cohort.parquet  (2021 rows, 33 cols)
Wrote derived/anzpicr_cohort.parquet (175 rows, 30 cols)
